# Cyberbullying Detection — Kaggle Training Notebook

**Run every cell from top to bottom in order.**

## Before you start — Upload your project as a Dataset

1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets)
2. Click **New Dataset** (top-right)
3. Upload your `cyberbulling-detection.zip` file
4. Name it `cyberbullying-detection` and click **Create**
5. Wait for it to finish processing (1–2 minutes)
6. Come back to this notebook and click **+ Add Data** (top-right) → search for your dataset → **Add**
7. Make sure GPU is enabled: **Settings (right panel) → Accelerator → GPU T4 x2 or P100**
8. Run the cells below

## What this notebook does
| Step | Action |
|---|---|
| 1 | Verify GPU is available |
| 2 | Extract your project from the Kaggle dataset |
| 3 | Install dependencies |
| 4 | Fix dataset (drop null row, create placeholder image) |
| 5 | Train with **m-BERT** |
| 6 | Evaluate m-BERT |
| 7 | Train with **MuRIL** |
| 8 | Evaluate MuRIL |
| 9 | Save all outputs (auto-saved to Kaggle output tab) |

---
## Step 1 — Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu}  ({mem:.1f} GB VRAM)')
else:
    print('No GPU detected.')
    print('Go to Settings (right panel) → Accelerator → GPU P100 or T4 x2')
    raise SystemExit('GPU required — enable it in Settings then restart the notebook')

---
## Step 2 — Load Your Project

Kaggle **auto-extracts** datasets — there is no zip file to unzip.
This cell finds the project files in `/kaggle/input/` and copies them to `/kaggle/working/` (which is writable).

In [ ]:
import os, glob, shutil, sys

INPUT_DIR = '/kaggle/input'
WORK_DIR  = '/kaggle/working'

# Show everything Kaggle has mounted so we can see the exact path
print('All mounted input paths:')
for root, dirs, files in os.walk(INPUT_DIR):
    level = root.replace(INPUT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level >= 2:
        for f in files:
            print(f'{indent}  {f}')
    dirs[:] = dirs if level < 3 else []

In [ ]:
# Kaggle auto-extracts datasets — find the folder that contains src/
INPUT_PROJECT_DIR = None
for root, dirs, files in os.walk(INPUT_DIR):
    if 'src' in dirs and 'requirements.txt' in files:
        INPUT_PROJECT_DIR = root
        break

if INPUT_PROJECT_DIR is None:
    # Fallback: any folder that has src/
    for root, dirs, files in os.walk(INPUT_DIR):
        if 'src' in dirs:
            INPUT_PROJECT_DIR = root
            break

if INPUT_PROJECT_DIR is None:
    raise FileNotFoundError(
        'Could not find the project under /kaggle/input/.\n'
        'Make sure you clicked "+ Add Input" on the right panel and added your dataset.'
    )

print(f'Found project at: {INPUT_PROJECT_DIR}')
print('Contents:', os.listdir(INPUT_PROJECT_DIR))

# Copy to /kaggle/working/ — input directory is READ-ONLY on Kaggle
PROJECT_DIR = os.path.join(WORK_DIR, 'project')
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
shutil.copytree(INPUT_PROJECT_DIR, PROJECT_DIR)
print(f'\nCopied to: {PROJECT_DIR}')

# Set working directory and Python path
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f'Working directory: {os.getcwd()}')

---
## Step 3 — Install Dependencies

Kaggle already has PyTorch, torchvision, pandas, scikit-learn, and Pillow.
Only `transformers` needs to be installed.

In [ ]:
!pip install -q transformers==4.40.0 accelerate -U
print('Dependencies installed')

---
## Step 4 — Fix Dataset

In [ ]:
import pandas as pd
from PIL import Image

# Fix 1: Drop the 1 null text_content row
df = pd.read_csv('data/train.csv')
df_clean = df.dropna(subset=['text_content']).reset_index(drop=True)
df_clean.to_csv('data/train_clean.csv', index=False)
print(f'Original: {len(df):,} rows  →  After cleaning: {len(df_clean):,} rows')

# Fix 2: Create placeholder image
os.makedirs('data/images', exist_ok=True)
if not os.path.exists('data/images/none.jpg'):
    Image.new('RGB', (224, 224), color=(128, 128, 128)).save('data/images/none.jpg')
print('Placeholder image: data/images/none.jpg created')

# Label distribution check
print('\nLabel distribution:')
for col in ['aggression', 'repetition', 'intent']:
    pos = int(df_clean[col].sum())
    print(f'  {col:12s}: {pos}/{len(df_clean)} positive ({pos/len(df_clean)*100:.1f}%)')

In [ ]:
# Point config to the clean CSV
import importlib
import src.config as cfg

cfg.TRAIN_CSV = os.path.join(cfg.DATA_DIR, 'train_clean.csv')

print(f'TRAIN_CSV : {cfg.TRAIN_CSV}')
print(f'IMAGE_DIR : {cfg.IMAGE_DIR}')
print(f'DEVICE    : {cfg.DEVICE}')
print(f'TEXT_MODEL: {cfg.TEXT_MODEL}')
print(f'EPOCHS    : {cfg.EPOCHS}')
print(f'BATCH_SIZE: {cfg.BATCH_SIZE}')

---
## Step 5 — Train with m-BERT

**Expected time on Kaggle P100: ~3–6 minutes per epoch**

Watch the output for:
- `Val Loss` going down — training is working
- `Best model saved` — checkpoint written
- `Early stopping` — done early because val loss stopped improving (this is fine and expected)

In [ ]:
import src.train as train_module
importlib.reload(train_module)

print('=' * 60)
print('  TRAINING — m-BERT (bert-base-multilingual-cased)')
print('=' * 60)

mbert_model = train_module.train(
    csv_path=cfg.TRAIN_CSV,
    image_dir=cfg.IMAGE_DIR,
    text_model='bert-base-multilingual-cased'
)

In [ ]:
# Rename m-BERT checkpoints before MuRIL training overwrites them
for fname in ['best_model.pth', 'final_model.pth']:
    src_path = os.path.join('models', fname)
    dst_path = os.path.join('models', fname.replace('.pth', '_mbert.pth'))
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        size_mb = os.path.getsize(dst_path) / 1e6
        print(f'Saved: {dst_path} ({size_mb:.1f} MB)')

---
## Step 6 — Evaluate m-BERT

In [ ]:
import src.evaluate as eval_module
importlib.reload(eval_module)

# Temporarily set TEXT_MODEL for the tokenizer
cfg.TEXT_MODEL = 'bert-base-multilingual-cased'

print('=' * 60)
print('  EVALUATION — m-BERT')
print('=' * 60)

eval_module.evaluate(
    model_path=os.path.join('models', 'best_model_mbert.pth'),
    csv_path=cfg.TRAIN_CSV
)

# Copy report so it doesn't get overwritten
shutil.copy2('models/evaluation_report.txt', 'models/evaluation_report_mbert.txt')
print('\nReport saved: models/evaluation_report_mbert.txt')

---
## Step 7 — Train with MuRIL

MuRIL was trained on 17 South Asian languages including transliterated Urdu (i.e. Roman Urdu).
It should outperform m-BERT on this dataset.

**Note:** MuRIL (~900 MB) downloads automatically on first use.

In [ ]:
importlib.reload(train_module)

print('=' * 60)
print('  TRAINING — MuRIL (google/muril-base-cased)')
print('=' * 60)

muril_model = train_module.train(
    csv_path=cfg.TRAIN_CSV,
    image_dir=cfg.IMAGE_DIR,
    text_model='google/muril-base-cased'
)

In [ ]:
# Rename MuRIL checkpoints
for fname in ['best_model.pth', 'final_model.pth']:
    src_path = os.path.join('models', fname)
    dst_path = os.path.join('models', fname.replace('.pth', '_muril.pth'))
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        size_mb = os.path.getsize(dst_path) / 1e6
        print(f'Saved: {dst_path} ({size_mb:.1f} MB)')

---
## Step 8 — Evaluate MuRIL

In [ ]:
cfg.TEXT_MODEL = 'google/muril-base-cased'
importlib.reload(eval_module)

print('=' * 60)
print('  EVALUATION — MuRIL')
print('=' * 60)

eval_module.evaluate(
    model_path=os.path.join('models', 'best_model_muril.pth'),
    csv_path=cfg.TRAIN_CSV
)

shutil.copy2('models/evaluation_report.txt', 'models/evaluation_report_muril.txt')
print('\nReport saved: models/evaluation_report_muril.txt')

---
## Step 9 — List All Output Files

All files in `/kaggle/working/models/` are automatically saved to your Kaggle output.
After this cell runs, click **Output** (top-right) → **Download All** to get everything.

In [ ]:
models_dir = os.path.join(PROJECT_DIR, 'models')
print('Files in models/ directory:')
print('-' * 50)
total_size = 0
for fname in sorted(os.listdir(models_dir)):
    fpath = os.path.join(models_dir, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    total_size += size_mb
    print(f'  {fname:<40}  {size_mb:6.1f} MB')
print('-' * 50)
print(f'  Total: {total_size:.1f} MB')
print()
print('To download: click the Output tab (top-right) → Download All')

In [ ]:
# Print both evaluation reports side-by-side for your notes
for report_name in ['evaluation_report_mbert.txt', 'evaluation_report_muril.txt']:
    report_path = os.path.join(models_dir, report_name)
    if os.path.exists(report_path):
        model_name = 'm-BERT' if 'mbert' in report_name else 'MuRIL'
        print('=' * 60)
        print(f'  RESULTS — {model_name}')
        print('=' * 60)
        with open(report_path) as f:
            print(f.read())

---
## After downloading — What to do locally

1. Place the downloaded `.pth` files in your local `models/` folder:
   ```
   cyberbulling-detection/
   └── models/
       ├── best_model_mbert.pth
       ├── best_model_muril.pth
       ├── final_model_mbert.pth
       └── final_model_muril.pth
   ```

2. Run baseline comparisons locally:
   ```bash
   python -m src.compare_models
   python -m src.baselines
   ```

3. Launch the Streamlit demo:
   ```bash
   pip install streamlit
   streamlit run demo.py
   ```